[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/06_dpo_preference_pairs.ipynb)

# Step 6 — DPO Preference Pairs (Optional)

Generate **preference pairs** for Direct Preference Optimization (DPO) that teach
a small model *refusal vs engagement calibration* on the **SEC investor bulletin**
(scope-boundary document).

SFT copies a single target answer. DPO instead says: for the same question,
**this** answer is better than **that** one — the right signal when the failure is
a tradeoff (too helpful vs too silent, or inventing SEC authority).

This notebook evaluates that exact preference objective. It compares the unadapted
Hugging Face model from `SFT_BASE_MODEL` with its DPO adapter on held-out boundary
questions. A position-balanced judge chooses the response that is more correct and
appropriately scoped using only the supplied passage—no gold answer and no Ollama.

## Learning objectives
- Build boundary questions from SEC scope-boundary paragraphs (every notebook-01 split)
- Generate four labeled candidates per question (one chosen, three rejected)
- Hold out boundary questions before expanding the remainder into DPO rows
- Fine-tune and evaluate the same `SFT_BASE_MODEL` checkpoint
- Measure DPO usefulness with passage-bounded pairwise preference judgments

## Prerequisites

1. Run **notebook 01** so `data/paragraphs.jsonl` exists; this notebook uses it as source text only.
2. Configure the teacher and judge API in `implementations/qa_text_generation/.env`.
3. Set `SFT_BASE_MODEL` to the Hugging Face checkpoint to compare before and after DPO.
4. Install the `text-sft` dependency group and use an NVIDIA CUDA GPU for model inference/training.

The evaluation is self-contained within this notebook: it creates and saves its own held-out
boundary prompts and does not read notebook 01's test set, notebook 05's scores, or any Ollama output.
This notebook is **SEC-only**; it does not use the CFPB credit-card agreement.

In [4]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    PARAGRAPHS_PATH,
    Paragraph,
    create_judge_client,
    create_teacher_client,
    load_implementation_dotenv,
    load_typed_jsonl,
    save_typed_jsonl,
    use_repo_root,
    write_json,
)
from aieng.syn_data.text.dpo import (
    DEFAULT_DPO_QUESTIONS,
    DPO_ADAPTER_DIR,
    DPO_CANDIDATES_PATH,
    DPO_EVAL_PROMPTS_PATH,
    DPO_PAIRS_PATH,
    DPO_PREFERENCE_RESULTS_PATH,
    CalibrationPrompt,
    PreferencePair,
    candidates_to_dpo_pairs,
    evaluate_model_preferences,
    filter_pairs_with_judge,
    filter_sec_paragraphs,
    generate_boundary_prompts,
    generate_calibration_candidates,
    generate_preference_responses,
    split_calibration_prompts,
    summarize_preferences,
    summarize_rejected_kinds,
    train_lora_dpo,
)
from rich.console import Console
from rich.table import Table


load_implementation_dotenv()
use_repo_root(Path("."))

N_QUESTIONS = int(os.getenv("DPO_N_QUESTIONS", DEFAULT_DPO_QUESTIONS))
VALIDATE_WITH_JUDGE = os.getenv("VALIDATE_WITH_JUDGE", "0") == "1"
RUN_DPO = os.getenv("RUN_DPO", "0") == "1"
SFT_BASE_MODEL = os.getenv("SFT_BASE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")

console = Console(width=100)
console.print(
    f"N_QUESTIONS={N_QUESTIONS}  VALIDATE_WITH_JUDGE={VALIDATE_WITH_JUDGE}  "
    f"RUN_DPO={RUN_DPO}\nSFT_BASE_MODEL={SFT_BASE_MODEL}"
)

N_QUESTIONS=8  VALIDATE_WITH_JUDGE=False  RUN_DPO=False
SFT_BASE_MODEL=Qwen/Qwen2.5-3B-Instruct

## Why four candidates?

The SEC bulletin is **investor education** (passphrases, alerts, public Wi-Fi). It does
not tell anyone which stock to buy, and it does not turn optional tips into legal mandates.

For each boundary question we ask the teacher for **one JSON object** with four answers:

| Kind | DPO role | What it does wrong (or right) |
|------|----------|-------------------------------|
| `correctly_scoped` | **chosen** | Answers from the passage; hedges; refuses investment advice |
| `overreaching` | rejected | Gives buy/sell or personal advice the bulletin does not authorize |
| `underreaching` | rejected | Refuses a question the passage *could* answer |
| `authority_misattribution` | rejected | Claims the SEC requires or covers something it does not |

Each question expands to **three** DPO rows (chosen vs each rejected kind).

## 1. Load SEC paragraphs

Keep every SEC scope-boundary paragraph, including notebook 01's test split.
This notebook's evaluation holds out **generated questions**, not those test
paragraphs, so using them as source text does not leak the preference eval set.

Do **not** pass unfiltered `all_paragraphs` into generation: that file also
contains the CFPB credit-card agreement, which is out of scope here.

In [2]:
all_paragraphs = load_typed_jsonl(PARAGRAPHS_PATH, Paragraph.from_dict)
sec_paragraphs = filter_sec_paragraphs(all_paragraphs)

if not sec_paragraphs:
    raise FileNotFoundError(
        f"No SEC scope-boundary paragraphs in {PARAGRAPHS_PATH}. Run notebook 01 first."
    )

table = Table(title="SEC paragraphs (scope-boundary, all splits)")
table.add_column("#", justify="right")
table.add_column("para_id")
table.add_column("split")
table.add_column("chars", justify="right")
table.add_column("preview")
for i, paragraph in enumerate(sec_paragraphs[:8]):
    preview = paragraph.text.replace("\n", " ")[:80] + "…"
    table.add_row(
        str(i),
        paragraph.para_id,
        paragraph.split.value,
        str(len(paragraph.text)),
        preview,
    )
console.print(table)
console.print(f"Total SEC paragraphs: {len(sec_paragraphs)}")

                            SEC paragraphs (scope-boundary, all splits)                             
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ para_id                      ┃ split ┃ chars ┃ preview                                       ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0 │ sec_investor_bulletin::p0004 │ test  │   623 │ Use two-step verification or “multi-factor”   │
│   │                              │       │       │ authentication, if available. Your i…         │
│ 1 │ sec_investor_bulletin::p0005 │ test  │   346 │ Instead of a unique code sent to you by text  │
│   │                              │       │       │ message or email, however, some web…          │
│ 2 │ sec_investor_bulletin::p0006 │ test  │   335 │ Turn “on” account alerts. One of the easiest  │
│   │                              │       │       │ ways to protect your online investm…          │
│ 3 │ sec_investor_bulletin::p0009 │ test  │   477 │ Use different passwords for different         │
│   │                              │       │       │ accounts. Avoid using the same password fo…   │
│ 4 │ sec_investor_bulletin::p0012 │ test  │   466 │ Use caution with wireless (or “Wi-Fi”)        │
│   │                              │       │       │ connections. If you use a wireless connec…    │
│ 5 │ sec_investor_bulletin::p0013 │ test  │   302 │ If you access your account on a public        │
│   │                              │       │       │ wireless connection, such as at a coffee …    │
│ 6 │ sec_investor_bulletin::p0014 │ test  │   868 │ Do not type your password unless the website  │
│   │                              │       │       │ you are accessing uses a secure con…          │
│ 7 │ sec_investor_bulletin::p0015 │ test  │   726 │ Make sure the software and software           │
│   │                              │       │       │ application (apps) on all your mobile device… │
└───┴──────────────────────────────┴───────┴───────┴───────────────────────────────────────────────┘

Total SEC paragraphs: 21

## 2. Generate boundary questions

The teacher cycles three question types: **in-scope** (should answer with a hedge),
**out-of-scope** (should refuse investment advice), and **gray-boundary** (related to
the bulletin but easy to overclaim SEC authority).

Demo size is small (`N_QUESTIONS`, default 8). Increase it for a real corpus.

In [5]:
teacher = create_teacher_client()
boundary_prompts = generate_boundary_prompts(
    teacher, sec_paragraphs, n_questions=N_QUESTIONS
)

q_table = Table(title="Boundary questions")
q_table.add_column("id")
q_table.add_column("kind")
q_table.add_column("question")
for prompt in boundary_prompts:
    q_table.add_row(prompt.id, prompt.question_kind.value, prompt.question[:90])
console.print(q_table)

                                         Boundary questions                                         
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ id           ┃ kind          ┃ question                                                          ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ dpo-c6ee7839 │ in_scope      │ According to the passage, what is the difference between a        │
│              │               │ password and a passphrase, and                                    │
│ dpo-1ceb8f10 │ out_of_scope  │ I am setting up a third-party authenticator app for my brokerage  │
│              │               │ account as suggested in t                                         │
│ dpo-ce917ffd │ gray_boundary │ According to the SEC investor-education bulletin, does the SEC    │
│              │               │ legally mandate that all re                                       │
│ dpo-b1c899a2 │ in_scope      │ According to the bulletin, why is using a single password for     │
│              │               │ multiple online financial ac                                      │
│ dpo-98f914df │ out_of_scope  │ Based on the Wi-Fi security bulletin, I want to protect my online │
│              │               │ investment accounts. Whi                                          │
│ dpo-966544eb │ gray_boundary │ According to the SEC's investor bulletin, are public wireless     │
│              │               │ networks legally prohibited                                       │
│ dpo-190da109 │ in_scope      │ According to the passage, how can I determine if a website is     │
│              │               │ secure before typing my pass                                      │
│ dpo-fc991761 │ out_of_scope  │ Based on the bulletin's advice to keep software up-to-date and    │
│              │               │ choose the least amount of                                        │
└──────────────┴───────────────┴───────────────────────────────────────────────────────────────────┘

## 3. Generate four candidates per question

One teacher call returns all four answers so the contrast is internally consistent.

In [6]:
calibration_prompts: list[CalibrationPrompt] = []
for prompt in boundary_prompts:
    try:
        calibration_prompts.append(generate_calibration_candidates(teacher, prompt))
    except (KeyError, ValueError, TypeError, RuntimeError) as exc:
        console.print(f"[yellow]Skipping {prompt.id}: {type(exc).__name__}: {exc}[/yellow]")

example = next(
    (item for item in calibration_prompts if item.candidates),
    None,
)
if example is None:
    raise RuntimeError("Teacher returned no candidates. Check the API key and model.")

console.print(f"[bold]Example question[/bold] ({example.question_kind.value}):")
console.print(example.question)

cand_table = Table(title=f"Candidates for {example.id}")
cand_table.add_column("kind", style="cyan")
cand_table.add_column("role")
cand_table.add_column("answer preview")
cand_table.add_column("rationale")
for candidate in example.candidates:
    role = "chosen" if candidate.kind.value == "correctly_scoped" else "rejected"
    cand_table.add_row(
        candidate.kind.value,
        role,
        candidate.answer.replace("\n", " ")[:180],
        candidate.rationale[:180],
    )
console.print(cand_table)

Example question (in_scope):

According to the passage, what is the difference between a password and a passphrase, and how does 
the investment firm send you the unique code when you log in from an unrecognized computer?

                                    Candidates for dpo-c6ee7839                                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ kind                     ┃ role     ┃ answer preview              ┃ rationale                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ correctly_scoped         │ chosen   │ The provided passage does   │ This answer correctly        │
│                          │          │ not explain the difference  │ identifies that the passage  │
│                          │          │ between a password and a    │ lacks information on the     │
│                          │          │ passphrase. Regarding the   │ first part of the question,  │
│                          │          │ unique code, if you log in  │ accurately details the       │
│                          │          │ from an unrecognized        │ delivery methods for the     │
│                          │          │ computer, your investment   │ code, and properly hedges    │
│                          │          │ fi                          │ reg                          │
│ overreaching             │ rejected │ The passage doesn't define  │ This answer overreaches by   │
│                          │          │ the difference, but you     │ giving unauthorized personal │
│                          │          │ should always use a long    │ investment advice and making │
│                          │          │ passphrase instead of a     │ false guarantees about       │
│                          │          │ password to guarantee your  │ account safety.              │
│                          │          │ account is 100% safe from   │                              │
│                          │          │ hackers. To secure your ass │                              │
│ underreaching            │ rejected │ I cannot answer this        │ This answer underreaches by  │
│                          │          │ question because the text   │ completely refusing to       │
│                          │          │ does not define the         │ answer, ignoring the fact    │
│                          │          │ difference between a        │ that the passage does        │
│                          │          │ password and a passphrase,  │ contain the answer to the    │
│                          │          │ so I cannot provide any     │ second part of the user's    │
│                          │          │ information about account   │ question.                    │
│                          │          │ security proces             │                              │
│ authority_misattribution │ rejected │ The passage does not        │ This answer invents legal    │
│                          │          │ distinguish between a       │ mandates and enforcement     │
│                          │          │ password and a passphrase.  │ powers attributed to the SEC │
│                          │          │ However, the SEC legally    │ that are completely absent   │
│                          │          │ mandates that all           │ from the text.               │
│                          │          │ investment firms must send  │                              │
│                          │          │ this unique code to your    │                              │
│                          │          │ mobile device w             │                              │
└──────────────────────────┴──────────┴─────────────────────────────┴──────────────────────────────┘

## 4. Create a held-out preference evaluation split

Hold out one question of each available boundary kind **before** creating DPO rows.
Those questions and all four of their generated candidates are excluded from training.
The candidates provide metadata for the exercise, but evaluation never shows their
`correctly_scoped` answers to either model or to the judge.

The remaining prompts use the same passage-and-question instruction format for DPO training.

In [7]:
train_prompts, eval_prompts = split_calibration_prompts(calibration_prompts)
if not eval_prompts:
    raise RuntimeError(
        "Need at least two generated questions of a boundary kind to create a held-out evaluation."
    )

pairs = candidates_to_dpo_pairs(train_prompts)
kind_counts = summarize_rejected_kinds(pairs)

count_table = Table(title="Held-out split and DPO training rows")
count_table.add_column("item")
count_table.add_column("n", justify="right")
count_table.add_row("generated boundary questions", str(len(calibration_prompts)))
count_table.add_row("training questions", str(len(train_prompts)))
count_table.add_row("held-out evaluation questions", str(len(eval_prompts)))
for kind, count in kind_counts.items():
    count_table.add_row(f"training pairs: {kind}", str(count))
console.print(count_table)

if pairs:
    sample_pair = pairs[0]
    pair_table = Table(title=f"Example training pair {sample_pair.id}")
    pair_table.add_column("field", style="cyan")
    pair_table.add_column("text")
    pair_table.add_row("rejected_kind", sample_pair.rejected_kind.value)
    pair_table.add_row("chosen", sample_pair.chosen[:400])
    pair_table.add_row("rejected", sample_pair.rejected[:400])
    console.print(pair_table)

      Held-out split and DPO training rows      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━┓
┃ item                                     ┃ n ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━┩
│ generated boundary questions             │ 8 │
│ training questions                       │ 5 │
│ held-out evaluation questions            │ 3 │
│ training pairs: overreaching             │ 5 │
│ training pairs: underreaching            │ 5 │
│ training pairs: authority_misattribution │ 5 │
└──────────────────────────────────────────┴───┘

                          Example training pair dpo-c6ee7839-overreaching                           
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ field         ┃ text                                                                             ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ rejected_kind │ overreaching                                                                     │
│ chosen        │ The provided passage does not explain the difference between a password and a    │
│               │ passphrase. Regarding the unique code, if you log in from an unrecognized        │
│               │ computer, your investment firm will send it to either your e-mail or mobile      │
│               │ device, though the availability of this two-step verification process may vary   │
│               │ depending on your specific firm.                                                 │
│ rejected      │ The passage doesn't define the difference, but you should always use a long      │
│               │ passphrase instead of a password to guarantee your account is 100% safe from     │
│               │ hackers. To secure your assets, you must immediately buy shares in firms that    │
│               │ mandate mobile-device codes, as these are the only safe investments.             │
└───────────────┴──────────────────────────────────────────────────────────────────────────────────┘

## 5. Optional training-pair validation and save

Set `VALIDATE_WITH_JUDGE=1` to keep a training pair only when the judge prefers its
chosen answer over its rejected answer. This validates generated training data; it is
separate from the final base-vs-DPO evaluation.


In [ ]:
if VALIDATE_WITH_JUDGE:
    judge = create_judge_client()
    kept, dropped = filter_pairs_with_judge(judge, train_prompts, pairs)
    console.print(f"Judge kept {len(kept)} / {len(pairs)} pairs ({len(dropped)} dropped).")
    pairs = kept
else:
    console.print("Training-pair validation skipped (VALIDATE_WITH_JUDGE=0).")

save_typed_jsonl(
    DPO_CANDIDATES_PATH,
    train_prompts,
    to_dict=CalibrationPrompt.to_dict,
)
save_typed_jsonl(
    DPO_EVAL_PROMPTS_PATH,
    eval_prompts,
    to_dict=CalibrationPrompt.to_dict,
)
save_typed_jsonl(
    DPO_PAIRS_PATH,
    pairs,
    to_dict=PreferencePair.to_dict,
)
console.print(f"Wrote training candidates → {DPO_CANDIDATES_PATH}")
console.print(f"Wrote held-out prompts    → {DPO_EVAL_PROMPTS_PATH}")
console.print(f"Wrote filtered training pairs      → {DPO_PAIRS_PATH}")

Judge kept 22 / 24 pairs (2 dropped).

Wrote candidates → 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/synthetic/dpo_candidates
.jsonl

Wrote pairs      → 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/synthetic/dpo_preference
_pairs.jsonl

## 6. Generate the Hugging Face baseline, then run optional LoRA DPO

First load the unadapted Hugging Face checkpoint named by `SFT_BASE_MODEL` and generate
responses for this notebook's held-out prompts. These responses are the direct control
for DPO: same model family, inference stack, prompt, and decoding settings.

Then optionally train a LoRA DPO adapter from that exact checkpoint. Both steps require
CUDA with the current 4-bit implementation. Leave `RUN_DPO=0` if you only want to create
and inspect the preference datasets.

**Direct Preference Optimization (DPO)** trains on **pairs**. For the same prompt \(x\), the model should assign higher probability to the **chosen** completion \(y_w\) than to the **rejected** one \(y_l\).

Classic RLHF first fits a **reward model**, then runs **PPO**. DPO skips both — the ranking is a classification-style loss on the policy, with a frozen **reference** model $\pi_{\text{ref}}$ (usually the model checkpoint) so the policy does not collapse or drift.

| | **SFT** | **RLHF** | **DPO** |
|--|--|--|--|
| Signal | one gold \(y\) | preferences → reward → PPO | preferences → one loss |
| Extra models | — | reward model + critic | frozen reference only |


![DPO chart](./images/DPO.png)


**Loss hyper-parameters** $ \sigma $ = sigmoid, $\beta$ = how hard to stay near $\pi_{\text{ref}}$: 

$$
\mathcal{L}_{\text{DPO}} = -\log\sigma\Big(\beta \log\frac{\pi_\theta(y_w\mid x)}{\pi_{\text{ref}}(y_w\mid x)} - \beta \log\frac{\pi_\theta(y_l\mid x)}{\pi_{\text{ref}}(y_l\mid x)}\Big)
$$

TRL’s `DPOTrainer` implements this.

**References**
- Rafailov et al., *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*, NeurIPS 2023. [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)
- Hugging Face TRL — [DPO Trainer](https://huggingface.co/docs/trl/dpo_trainer)
- Ouyang et al., *Training language models to follow instructions with human feedback* (RLHF baseline). [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)


In [7]:
# Reload only artifacts created by this notebook, so evaluation can be rerun cleanly.
pairs = load_typed_jsonl(DPO_PAIRS_PATH, PreferencePair.from_dict)
eval_prompts = load_typed_jsonl(DPO_EVAL_PROMPTS_PATH, CalibrationPrompt.from_dict)
if not pairs or not eval_prompts:
    raise FileNotFoundError("Run sections 1–5 to create training pairs and held-out prompts.")
console.print(f"Training pairs: {len(pairs)}; held-out prompts: {len(eval_prompts)}")

Number of DPO pairs: 22


### Run the baseline model

In [8]:
from aieng.syn_data.text.sft import Hf4BitInferenceClient

if RUN_DPO:
    console.print(f"Loading Hugging Face baseline: {SFT_BASE_MODEL}")
    baseline_client = Hf4BitInferenceClient(SFT_BASE_MODEL)
    baseline_responses = generate_preference_responses(baseline_client, eval_prompts)
    baseline_client.release()
    console.print(f"Generated {len(baseline_responses)} held-out baseline responses.")
else:
    baseline_responses = {}
    console.print(
        "[yellow]Baseline inference skipped with RUN_DPO=0. "
        "Set RUN_DPO=1 on CUDA to run the before/after comparison.[/yellow]"
    )


Baseline inference skipped with RUN_DPO=0. Set RUN_DPO=1 on CUDA to run the before/after comparison.

### Train LoRA adaptor for the base model

In [9]:
if RUN_DPO:
    adapter_path = train_lora_dpo(
        pairs,
        DPO_ADAPTER_DIR,
        base_model=SFT_BASE_MODEL,
        num_train_epochs=1.0,
    )
    console.print(f"[bold green]DPO LoRA adapter saved to {adapter_path}[/bold green]")
else:
    console.print(
        "[bold yellow]DPO training skipped[/bold yellow]\n"
        "Set [green]RUN_DPO=1[/green] on a [green]CUDA[/green] machine to fine-tune.\n"
        f"Preference pairs are ready at {DPO_PAIRS_PATH}"
    )

DPO training skipped
Set RUN_DPO=1 on a CUDA machine to fine-tune.
Preference pairs are ready at 
/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/qa_text_generat
ion/DPO/synthetic/dpo_preference_pairs.jsonl

# Evaluation
## 7. Passage-bounded preference evaluation

Generate DPO responses on the held-out prompts, then compare each against the baseline
response. The judge sees the source passage, question type, question, and two anonymous
responses. It does **not** see the generated chosen answer.

Each comparison runs twice with response order reversed. **A model wins only when both
orders agree**; disagreement is conservatively recorded as a tie. The resulting win/tie
rate directly tests the DPO objective and exposes results by `in_scope`, `out_of_scope`,
and `gray_boundary` rather than averaging general-quality scores.


In [10]:
from aieng.syn_data.text.sft import PeftInferenceClient

adapter_ready = DPO_ADAPTER_DIR.exists() and any(DPO_ADAPTER_DIR.iterdir())
if RUN_DPO and adapter_ready:
    dpo_client = PeftInferenceClient(DPO_ADAPTER_DIR, SFT_BASE_MODEL)
    dpo_responses = generate_preference_responses(dpo_client, eval_prompts)
    dpo_client.release()
    console.print(f"Generated {len(dpo_responses)} held-out DPO responses.")
elif RUN_DPO:
    raise FileNotFoundError(
        f"No adapter at {DPO_ADAPTER_DIR}. Run the DPO training cell first."
    )
else:
    dpo_responses = {}


Test set: 56 total, 19 refusal_calibration

In [11]:
if RUN_DPO:
    preference_judge = create_judge_client()
    judgments = evaluate_model_preferences(
        preference_judge,
        eval_prompts,
        baseline_responses,
        dpo_responses,
    )
    preference_summary = summarize_preferences(judgments)

    report = {
        "base_model": SFT_BASE_MODEL,
        "adapter_path": str(DPO_ADAPTER_DIR),
        "evaluation_design": "passage_bounded_position_balanced_pairwise_preference",
        "uses_gold_answers": False,
        "summary": preference_summary,
        "judgments": [judgment.to_dict() for judgment in judgments],
        "responses": [
            {
                "prompt_id": prompt.id,
                "question_kind": prompt.question_kind.value,
                "question": prompt.question,
                "context": prompt.context,
                "baseline_response": baseline_responses[prompt.id],
                "dpo_response": dpo_responses[prompt.id],
            }
            for prompt in eval_prompts
        ],
    }
    write_json(DPO_PREFERENCE_RESULTS_PATH, report)

    result_table = Table(title="Does DPO improve boundary preferences?")
    result_table.add_column("question kind", style="cyan")
    result_table.add_column("n", justify="right")
    result_table.add_column("base wins", justify="right", style="yellow")
    result_table.add_column("DPO wins", justify="right", style="green")
    result_table.add_column("ties", justify="right")
    result_table.add_column("DPO preference", justify="right", style="magenta")
    rows = {"overall": preference_summary["overall"], **preference_summary["by_question_kind"]}
    for kind, values in rows.items():
        result_table.add_row(
            kind,
            str(values["n"]),
            str(values["baseline_wins"]),
            str(values["dpo_wins"]),
            str(values["ties"]),
            f'{values["dpo_preference_rate"]:.1%}',
        )
    console.print(result_table)

    overall = preference_summary["overall"]
    if overall["dpo_wins"] > overall["baseline_wins"]:
        console.print("[bold green]DPO is preferred more often than the baseline on this held-out set.[/bold green]")
    else:
        console.print("[bold yellow]This run does not show a preference advantage over the baseline.[/bold yellow]")
    console.print(f"Wrote detailed preference report → {DPO_PREFERENCE_RESULTS_PATH}")
else:
    console.print("Preference evaluation skipped because RUN_DPO=0.")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 713.16it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-08-27 22:50:48,609 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-sec_investor_bulletin::p0004-0 (answer length: 927)
2026-08-27 22:50:50,081 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-sec_investor_bulletin::p0004-2 (answer length: 599)
2026-08-27 22:50:51,083 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-sec_investor_bulletin::p0005-0 (answer length: 112)
2026-08-27 22:50:52,345 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-sec_investor_bulletin::p0005-2 (answer length: 649)
2026-08-27 22:50:53,513 INFO aieng.syn_data.text.judge: Scoring model answer for sample: test-sec_investor_bulletin::p0006-0 (answer length: 120)
2026-08-27 22:50:54,451 INFO aieng.syn_data.text.judg

 DPO LoRA — refusal_calibration  
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Metric                ┃ Score ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ correctness           │ 3.421 │
│ coherence             │ 4.789 │
│ instruction_following │ 4.737 │
│ factual_plausibility  │ 3.895 │
│ average               │ 4.211 │
└───────────────────────┴───────┘

                Refusal calibration: Ollama vs HF base vs DPO                 
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Metric                ┃ Ollama ┃ HF base ┃ DPO LoRA ┃ Δ Ollama ┃ Δ HF base ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━┩
│ correctness           │  4.053 │   3.553 │    3.421 │   -0.632 │    -0.132 │
│ coherence             │  4.895 │   4.789 │    4.789 │   -0.105 │    +0.000 │
│ instruction_following │  4.684 │   4.789 │    4.737 │   +0.053 │    -0.053 │
│ factual_plausibility  │  4.500 │   4.000 │    3.895 │   -0.605 │    -0.105 │
│ average               │  4.533 │   4.283 │    4.211 │   -0.322 │    -0.072 │
└───────────────────────┴────────┴─────────┴──────────┴──────────┴───────────┘

Wrote 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/results/dpo_refusal_pred
ictions.jsonl

Wrote 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/results/dpo_refusal_scor
es.json